# Gemma 3 fine tune

In [3]:
# ============================================================================
# DEPENDENCIES: Install required packages
# ============================================================================
# These packages handle:
# - transformers: Hugging Face library for LLMs
# - peft: Parameter-Efficient Fine-Tuning (LoRA support)
# - datasets: Loading and processing datasets from HuggingFace
# - torch: Deep learning framework (PyTorch)
# - accelerate: Multi-GPU/TPU distributed training
# - bitsandbytes: 8-bit quantization for memory efficiency
# ============================================================================

# Uncomment the line below if any packages are missing
# %pip install transformers peft datasets torch accelerate bitsandbytes

import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

print("All required dependencies are installed!")

All required dependencies are installed!


## Loading transformer

In [4]:
# Load gemma-2b from moldels directory to cuda
model_path = "models/gemma-2b"
if os.path.exists(model_path):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(model_path, quantization_config=bnb_config, device_map="cuda")
    model.to("cuda")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    print("Model loaded to CUDA successfully!")
else:
    print(f"Model path '{model_path}' does not exist. Please ensure the model is downloaded and placed in the correct directory.")

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Model loaded to CUDA successfully!


In [5]:
prompt = """
World war 2
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


World war 2
The world war 2 was a war between the two great powers of the world. The war was fought between the United States and Germany. The war was fought from 1941 to 1945. The war was fought between the United States and Germany. The war was fought between the United States and Germany. The war was fought between the United States and Germany. The war was fought between the United States and Germany. The war was fought between the United States and Germany.


## Alpaca fine-Tuning

In [6]:
# ============================================================================
# ALPACA FINE-TUNING WITH GEMMA 2B - COMPREHENSIVE LEARNING GUIDE
# ============================================================================
# This section demonstrates parameter-efficient fine-tuning using LoRA (Low-Rank Adaptation)
# Why LoRA? It reduces trainable parameters from millions to thousands, enabling training on
# consumer GPUs. We freeze base model weights and add small trainable "adapter" matrices.
# ============================================================================

import torch
from datasets import load_dataset, load_from_disk

# Step 1: Download Alpaca dataset from HuggingFace
# The Alpaca dataset contains 52k instruction-following examples in the format:
# {"instruction": "...", "input": "...", "output": "..."}
# This format teaches the model to follow instructions, not just predict text.
print("Loading Alpaca dataset...")

if os.path.exists('datasets/alpaca_split'):
    print("Alpaca dataset already exists locally. Loading from disk...")
    dataset = load_from_disk('datasets/alpaca_split')
else:
    dataset = load_dataset("tatsu-lab/alpaca")
    dataset = dataset["train"].train_test_split(
        test_size=0.1,
        seed=42
    )

# Examine the structure of one example
print("\nDataset structure:")
print(dataset)
print(f"Total examples: {len(dataset['train'])}")
print("\nFirst example:")
print(dataset['train'][0])

Loading Alpaca dataset...
Alpaca dataset already exists locally. Loading from disk...

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 46801
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 5201
    })
})
Total examples: 46801

First example:
{'instruction': 'What is a digital identity and why is it important?', 'input': '', 'output': "A digital identity is a set of online identifiers and characteristics that can be used to identify a person or online entity. It's important because it is linked to all online activities, from financial transactions to communication, and is used to protect personal information and data while maintaining privacy.", 'text': "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat is a digital identity and why is it important?\n\n### Respons

## Step 2: FORMAT DATASET FOR INSTRUCTION-TUNING

In [22]:
# ============================================================================
# STEP 2: FORMAT DATASET FOR INSTRUCTION-TUNING
# ============================================================================
# Why formatting matters: Models need consistent input format. We'll create
# instruction prompts with the format: "### Instruction:\n...\n### Response:\n..."
# This teaches the model where instructions end and responses begin.
# ============================================================================

def format_instruction(example):
    """
    Converts raw Alpaca examples into instruction-response format.
    
    Why this format?
    - Clear separation helps model learn what is instruction vs. output
    - Mimics the format Alpaca was originally trained with
    - Improves model's ability to follow diverse instructions
    
    Args:
        example (dict): Single dataset example with 'instruction', 'input', 'output'
    
    Returns:
        dict: Example with 'text' field containing formatted prompt
    """
    # Extract fields from the example
    instruction = example['instruction']
    input_text = example['input']
    output_text = example['output']
    
    # Build the formatted prompt
    # The ### markers help the model learn to recognize instruction boundaries
    if input_text:
        # Use input field if provided (for context-aware instructions)
        # Example: instruction="Summarize this text", input="The quick brown fox..."
        prompt = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{output_text}"""
    else:
        # Simpler format for instructions without additional input
        # Example: instruction="What is 2+2?", response="4"
        prompt = f"""### Instruction:
{instruction}

### Response:
{output_text}"""
    
    # Add EOS token to indicate end of sequence
    prompt += tokenizer.eos_token
    
    # Return as 'text' field expected by transformers
    return {'text': prompt}
# End of format_instruction function definition


# Reload alpaca dataset from disk incase format_instruction is modified
dataset = load_from_disk('datasets/alpaca_split')

# Apply formatting to all examples
# This creates a new column 'text' without modifying the original data
dataset = dataset.map(format_instruction, remove_columns=['instruction', 'input', 'output'])

print("First formatted example:")
print(dataset['train'][0]['text'])
print("\n" + "="*80 + "\n")

Map:   0%|          | 0/46801 [00:00<?, ? examples/s]

Map:   0%|          | 0/5201 [00:00<?, ? examples/s]

First formatted example:
### Instruction:
What is a digital identity and why is it important?

### Response:
A digital identity is a set of online identifiers and characteristics that can be used to identify a person or online entity. It's important because it is linked to all online activities, from financial transactions to communication, and is used to protect personal information and data while maintaining privacy.<eos>




## Step 3: Set up LoRA

In [8]:
# ============================================================================
# STEP 3: SET UP LoRA (LOW-RANK ADAPTATION) FOR PARAMETER-EFFICIENT FINE-TUNING
# ============================================================================
# Why LoRA instead of full fine-tuning?
# - Full fine-tuning: ~2.5B parameters × 4 bytes = 10GB+ VRAM needed
# - LoRA fine-tuning: ~100K parameters × 4 bytes = ~1MB VRAM needed
# - LoRA works by adding small "adapter" matrices (A and B) to key layers
# - Mathematical principle: Weight updates ≈ Low-rank matrices (often true in practice)
# - Training speed: ~3-5x faster than full fine-tuning
# ============================================================================

from peft import LoraConfig, get_peft_model

# Create LoRA configuration
lora_config = LoraConfig(
    # r: Rank of the low-rank matrices
    # Why smaller r (8-16) for small models? 
    # - Gemma 2B is compact, doesn't need high-rank updates
    # - r=8 means adding 8-dimensional subspaces to weight matrices
    # - Too high r wastes parameters, too low r limits expressiveness
    r=8,
    
    # lora_alpha: Scaling factor for LoRA updates
    # Why 32 (2x r)?
    # - Prevents training instability by controlling gradient magnitude
    # - Typical ratio: lora_alpha = 2 * r
    # - Higher alpha = larger learning signal but can cause divergence
    lora_alpha=16,
    
    # target_modules: Which layers to apply LoRA to
    # Why only ["q_proj", "v_proj"]?
    # - These are Query and Value projection matrices in attention layers
    # - Attention is where models learn to focus on relevant information
    # - Full LoRA would include k_proj too, but these two are most important
    # - Limiting targets speeds up training while maintaining effectiveness
    target_modules=["q_proj", "v_proj"],
    
    # lora_dropout: Dropout probability on LoRA weights
    # Why 0.05?
    # - Small dropout prevents overfitting on the small Alpaca dataset
    # - Unlike standard dropout (0.1), LoRA uses less because
    #   the weights are already low-rank (inherent regularization)
    lora_dropout=0.05,
    
    # bias: How to handle bias terms
    # "none" = don't add LoRA to bias vectors (only to weight matrices)
    # This is standard practice and reduces parameters further
    bias="none",
    
    # task_type: Type of task for proper module handling
    # "CAUSAL_LM" means next-token prediction (language modeling)
    task_type="CAUSAL_LM",
)

# Apply LoRA to the model
# This wraps the original model and adds trainable adapter modules
# The base model parameters remain frozen (not updated during training)
model = get_peft_model(model, lora_config)

# Print trainable parameter count
# This shows how effective LoRA is at parameter reduction
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {all_params:,}")
print(f"Trainable ratio: {100 * trainable_params / all_params:.2f}%")
print(f"\nBefore LoRA, full fine-tuning would require {all_params/1e9:.2f}B gradients")
print(f"With LoRA, we only train {trainable_params/1e6:.2f}M parameters!")
print("\n" + "="*80 + "\n")

Trainable parameters: 921,600
Total parameters: 1,516,189,696
Trainable ratio: 0.06%

Before LoRA, full fine-tuning would require 1.52B gradients
With LoRA, we only train 0.92M parameters!




## Step 4: Tokenize dataset

In [9]:
# ============================================================================
# STEP 4: TOKENIZE DATASET AND PREPARE FOR TRAINING
# ============================================================================
# Tokenization: Converting text → numbers that the model understands
# Each token is typically a subword (common piece of text)
# Alpaca examples vary in length, so we need careful handling
# ============================================================================

def tokenize_function(example):
    """
    Tokenize text and prepare for training.
    
    Why this tokenization approach?
    - We tokenize the full instruction+response together
    - This teaches the model the joint distribution of (instruction, response)
    - max_length=512: Balances memory usage vs. capturing full context
    - KO: We reduce to 256 for faster training and lower VRAM usage, but you can increase if you have more resources
    - truncation=True: Longer examples are cut off (rare in Alpaca, ~95% < 512 tokens)
    - padding=True: Shorter examples padded with special [PAD] token
    
    Args:
        example (dict): Dataset example with 'text' field
    
    Returns:
        dict: Tokenized with 'input_ids', 'attention_mask', 'labels'
    """
    # Tokenize the text
    # return_tensors="pt" returns PyTorch tensors
    tokenized = tokenizer(
        example['text'],
        max_length=128,  # Gemma uses 8K context, but we start with 512 for efficiency
        truncation=True,  # Cut off overly long examples
        padding='max_length',  # Pad all to same length (required for batching)
    )
    
    # For causal language modeling (next-token prediction):
    # - input_ids: The tokens to feed as input
    # - labels: The expected next tokens (used for loss computation)
    # - We shift by 1: teach model to predict token[i+1] given token[i]
    # Why shift? If input is [A, B, C, D], labels should be [B, C, D, E]
    # This is how models learn to predict the next token
    
    tokenized['labels'] = tokenized['input_ids'].copy()
    
    return tokenized

# Apply tokenization to all examples
# This is expensive (~1-2 min for 52K examples), so we cache with batched=True
print("Tokenizing dataset (this may take 1-2 minutes)...")
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=False,  # Process one example at a time for clarity
    remove_columns=['text'],  # Remove original text column to save memory
    num_proc=4,  # Use 4 CPU cores for parallel processing (faster)
)

print("Tokenization complete!")
print(f"Dataset shape: {tokenized_dataset['train'].column_names}")
print(f"\nExample token count:")
print(f"  input_ids length: {len(tokenized_dataset['train'][0]['input_ids'])}")
print(f"  Sample tokens: {tokenized_dataset['train'][0]['input_ids'][:20]}")
print("\n" + "="*80 + "\n")

Tokenizing dataset (this may take 1-2 minutes)...
Tokenization complete!
Dataset shape: ['input_ids', 'attention_mask', 'labels']

Example token count:
  input_ids length: 128
  Sample tokens: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]




## Step 5. Create data collator for dynamic batching

In [10]:
# ============================================================================
# STEP 5: CREATE DATA COLLATOR FOR DYNAMIC BATCHING
# ============================================================================
# Why a data collator?
# - Converts list of examples → single batch tensor for GPU
# - Handles padding masking: tells model to ignore [PAD] tokens
# - Handles label masking: prevents loss computation on [PAD] tokens
# - Dynamic batching: pads only to longest sequence in batch (saves memory)
# ============================================================================

from transformers import DataCollatorForLanguageModeling

# Create data collator specifically for language modeling
data_collator = DataCollatorForLanguageModeling(
    # tokenizer: Used to get special token IDs (especially pad token)
    tokenizer=tokenizer,
    
    # mlm=False: We're doing causal LM (next-token prediction), not masked LM
    # Why causal? 
    # - Gemma is an autoregressive model (generates left-to-right)
    # - Causal masking: model can't see future tokens
    # - This is how LLMs work in practice
    mlm=False,
)

print(f"Data collator configured!")
print(f"Pad token ID: {tokenizer.pad_token_id}")
print(f"Pad token: '{tokenizer.decode([tokenizer.pad_token_id])}'")
print("\n" + "="*80 + "\n")

Data collator configured!
Pad token ID: 0
Pad token: '<pad>'




## Step 6. Training Arguments

In [23]:
# Minimum configuration for training
# 192m ~ 3h12m

# Import Hugging Face training configuration class
from transformers import TrainingArguments


# Configure training behavior
training_args = TrainingArguments(

    # Folder where checkpoints/logs will be saved
    output_dir="./gemma-2b-alpaca-lora",

    # Number of full passes through dataset
    #
    # 1 epoch is enough for first learning experiments
    #
    # Faster iteration
    # Lower overfitting risk
    num_train_epochs=1,

    # Batch size PER GPU STEP
    #
    # Very important for VRAM usage
    #
    # 1 is safest for 8GB GPU
    per_device_train_batch_size=1,

    # Evaluation batch size
    #
    # Evaluation uses less memory than training
    # but keep conservative initially
    per_device_eval_batch_size=1,

    # Simulate larger effective batch size
    #
    # Effective batch:
    # 1 * 4 = 4
    #
    # Helps gradient stability without large VRAM usage
    gradient_accumulation_steps=4,

    # Enable gradient checkpointing
    #
    # Recomputes activations during backward pass
    # instead of storing everything
    #
    # Lower VRAM
    # Slower training
    gradient_checkpointing=True,
    # Try this if you have enough VRAM and want faster training
    # gradient_checkpointing=False,

    # Learning rate for LoRA training
    #
    # LoRA commonly uses higher LR than full fine-tuning
    learning_rate=2e-4,

    # Small warmup helps stabilize early training
    warmup_steps=50,

    # Small regularization to reduce overfitting
    weight_decay=0.01,

    # Mixed precision training
    #
    # fp16 reduces VRAM usage significantly
    fp16=True,

    # Evaluate once per epoch
    #
    # Simpler and lighter than frequent step evals
    eval_strategy="epoch",

    # Save checkpoints once per epoch
    save_strategy="epoch",

    # Print logs every 10 steps
    logging_steps=10,

    # Remove unused dataset columns automatically
    remove_unused_columns=True,

    # Use memory-efficient optimizer
    #
    # Important for consumer GPUs
    optim="paged_adamw_8bit",

    # Disable external logging systems initially
    #
    # Simpler for learning/debugging
    report_to="none",

    # Reproducibility
    seed=42,
)

In [30]:
# ============================================================================
# STEP 6: CONFIGURE TRAINING PARAMETERS
# ============================================================================
# Training hyperparameters control how the model learns from data
# Choosing these well is crucial for good performance
# ============================================================================

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    # output_dir: Where to save model checkpoints and logs
    # We save intermediate models so we can restore if training fails
    output_dir="./gemma-2b-alpaca-lora",
    
    # learning_rate: How much to update weights per step
    # Why 3e-4 (0.0003)?
    # - Too high (e.g., 1e-3): Model updates too fast → training becomes unstable
    # - Too low (e.g., 1e-5): Model learns too slowly → requires more epochs
    # - For LoRA fine-tuning on smaller models: 3e-4 is a good sweet spot
    # - This is higher than base model training (typically 5e-5) because we're 
    #   only updating a small fraction of parameters
    learning_rate=3e-4,
    
    # num_train_epochs: How many times to iterate through the entire dataset
    # Why 3 epochs?
    # - 1 epoch: Usually under-fitting (model hasn't learned patterns yet)
    # - 3 epochs: Good balance for small datasets like Alpaca
    # - >5 epochs: Risk of overfitting (memorizing training data)
    num_train_epochs=1,
    
    # per_device_train_batch_size: Examples processed before updating weights
    # Why 8?
    # - Larger batches = more stable gradients but more GPU memory
    # - Smaller batches = noisier gradients but learn faster
    # - Gemma 2B + LoRA on most consumer GPUs: batch size 4-16 is safe
    # - We use 8 for decent speed and gradient stability
    # - KO: If you get OOM errors, reduce to 4 or use gradient accumulation (next parameter)
    per_device_train_batch_size=4,
    
    # per_device_eval_batch_size: Batch size for evaluation (less memory needed)
    # Why larger than training?
    # - Evaluation doesn't compute gradients (save ~2x memory)
    # - Larger eval batches = faster evaluation
    per_device_eval_batch_size=16,
    
    # gradient_accumulation_steps: Accumulate gradients over N steps before updating
    # Why 4?
    # - Effective batch size = per_device_train_batch_size * gradient_accumulation_steps
    # - Effective batch = 8 * 4 = 32 (better gradient stability)
    # - This technique allows training with bigger effective batches on limited GPUs
    gradient_accumulation_steps=4,
    
    # warmup_steps: Gradually increase learning rate at start
    # Why 100?
    # - Prevents unstable training from random initialization
    # - Learning rate goes: 0 → target_lr over 100 steps
    # - Formula: lr = initial_lr * (current_step / warmup_steps)
    # - ~5% of total steps for nice convergence
    warmup_steps=100,
    
    # weight_decay: L2 regularization penalty
    # Why 0.01?
    # - Prevents weights from growing too large
    # - Helps prevent overfitting on small dataset
    # - Typical range: 0.01-0.1
    weight_decay=0.01,
    
    # save_strategy: When to save model checkpoints
    # Why "steps"?
    # - "steps": Save every save_steps iterations (gives intermediate checkpoints)
    # - "epoch": Save once per epoch (can't recover mid-epoch if training fails)
    # - "no": Don't save (risky for long training!)
    save_strategy="steps",
    
    # save_steps: Save checkpoint every N training steps
    # Why 100?
    # - Too frequent (e.g., 10): Too many checkpoints → disk space issues
    # - Too infrequent (e.g., 500): Lose progress if training crashes
    # - Every 100 steps = ~10 checkpoints per epoch (good balance)
    save_steps=100,
    
    # eval_strategy: When to evaluate on validation set
    # Why "steps"?
    # - Monitor performance during training (not just at end)
    # - Helps detect overfitting early
    eval_strategy="steps",
    
    # eval_steps: Run evaluation every N steps
    # Why 100?
    # - Should match save_steps for checkpoint management
    # - More frequent evaluation = better monitoring but slower training
    eval_steps=100,
    
    # logging_steps: Log training metrics every N steps
    # Why 10?
    # - Frequent enough to see training progress
    # - Frequent logging helps debug training issues
    logging_steps=10,
    
    # report_to: Where to send metrics
    # Why "tensorboard"?
    # - TensorBoard: Free, local, great for monitoring training
    # - W&B: More features but requires account
    # - "none": Don't log (harder to debug)
    report_to="none",
    
    # seed: Random seed for reproducibility
    # Why 42?
    # - Ensures same shuffled data order across runs
    # - \"Answer to Life, Universe, and Everything\" (Douglas Adams reference 😄)
    # - Any number works; different seeds = slightly different results
    seed=42,
    
    # remove_unused_columns: Only pass needed columns to model
    # Why True?
    # - Saves memory and avoids unexpected input issues
    remove_unused_columns=True,
    
    # optim: Optimizer algorithm
    # Why \"paged_adamw_8bit\"?
    # - AdamW: Adaptive learning rate (converges faster than SGD)
    # - 8bit: Quantized to 8-bit precision (saves memory, minimal quality loss)
    # - paged: Uses virtual memory (handles large models on small GPUs)
    optim="paged_adamw_8bit",
)

# print(\"Training configuration:\")\nprint(f\"  Learning rate: {training_args.learning_rate}\")\nprint(f\"  Batch size (effective): {8 * 4} = {training_args.per_device_train_batch_size} * {training_args.gradient_accumulation_steps}\")\nprint(f\"  Epochs: {training_args.num_train_epochs}\")\nprint(f\"  Warmup steps: {training_args.warmup_steps}\")\nprint(f\"  Total steps per epoch: ~{len(tokenized_dataset['train']) // (8*4)}\")\nprint(\"\\n\" + \"=\"*80 + \"\\n\")
print("=" * 80)
print("Training Configuration")
print("=" * 80)

print(f"Learning rate:           {training_args.learning_rate}")
print(f"Batch size per device:   {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation:   {training_args.gradient_accumulation_steps}")
print(f"Effective batch size:    "
      f"{training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

print(f"Epochs:                  {training_args.num_train_epochs}")
print(f"Warmup steps:            {training_args.warmup_steps}")

steps_per_epoch = len(tokenized_dataset['train']) // (
    training_args.per_device_train_batch_size *
    training_args.gradient_accumulation_steps
)

print(f"Approx steps per epoch:  {steps_per_epoch}")

print("=" * 80)

Training Configuration
Learning rate:           0.0003
Batch size per device:   4
Gradient accumulation:   4
Effective batch size:    16
Epochs:                  3
Warmup steps:            100
Approx steps per epoch:  2925


## Step 7. Start training

In [24]:
# ============================================================================
# STEP 7: CREATE TRAINER AND START TRAINING
# ============================================================================
# The Trainer class handles the entire training loop:
# - Loading batches from dataset
# - Computing loss and gradients
# - Updating model weights
# - Saving checkpoints
# - Evaluating on validation data
# ============================================================================

from transformers import Trainer

# Create the Trainer
# Trainer is a high-level abstraction that handles all the training machinery
# It's much simpler than writing your own training loop, which is error-prone
trainer = Trainer(
    # model: The model to fine-tune (with LoRA adapters attached)
    model=model,
    
    # args: Training configuration (what we just defined above)
    args=training_args,
    
    # train_dataset: Data for training
    # Will iterate through this dataset multiple times (num_train_epochs)
    train_dataset=tokenized_dataset['train'],
    
    # eval_dataset: Data for validation (not in training data)
    # Used to monitor for overfitting during training
    eval_dataset=tokenized_dataset['test'],
    
    # data_collator: Converts list of examples → GPU batches
    # Handles tokenization edge cases and padding
    data_collator=data_collator,
)

print("Starting training...")
print("This will take some time (~30-60 min on consumer GPU)")
print("Monitor progress with: tensorboard --logdir ./gemma-2b-alpaca-lora/runs")
print("\n")

# Train the model
# This is where the actual fine-tuning happens
# During training, the model learns to follow instructions by predicting responses
trainer.train()

print("\nTraining complete!")
print(f"Model and LoRA adapters saved in: ./gemma-2b-alpaca-lora")

Starting training...
This will take some time (~30-60 min on consumer GPU)
Monitor progress with: tensorboard --logdir ./gemma-2b-alpaca-lora/runs




Epoch,Training Loss,Validation Loss
1,1.701944,1.744707



Training complete!
Model and LoRA adapters saved in: ./gemma-2b-alpaca-lora


In [25]:
# ============================================================================
# STEP 8: TEST THE FINE-TUNED MODEL
# ============================================================================
# Now let's test the fine-tuned model on new instructions
# This shows if the model learned to follow the Alpaca instruction format
# ============================================================================

def generate_response(instruction, input_text=""):
    """
    Generate a response using the fine-tuned model.
    
    Args:
        instruction (str): The instruction to follow
        input_text (str): Optional context for the instruction
    
    Returns:
        str: Generated response
    """
    # Format input in the same Alpaca style we trained on
    # This is important - the model expects this format!
    if input_text:
        prompt = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
"""
    else:
        prompt = f"""### Instruction:
{instruction}

### Response:
"""
    
    # Tokenize the prompt
    # We don't use return_tensors="pt" because we need to handle variable lengths
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Generate response
    # max_new_tokens=256: Generate up to 256 new tokens (usually 1-2 sentences)
    # temperature=0.7: Lower temp = more deterministic (0.0 = always pick best)
    #                Higher temp = more random (1.0 = uniform distribution)
    # top_p=0.9: Nucleus sampling - keep top 90% probability mass
    #           Helps avoid nonsense while maintaining diversity
    with torch.no_grad():  # Don't compute gradients (we're not training)
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,  # Use sampling instead of greedy decoding
        )
    
    # Decode the generated tokens back to text
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the response part (remove the prompt we added)
    # This makes output cleaner
    response = response.split("### Response:")[1].strip()
    
    return response

# Test the fine-tuned model
print("Testing fine-tuned Gemma 2B on Alpaca instructions!")
print("=" * 80)
print()

# Test 1: Simple factual question
instruction_1 = "What is EOS token in LLMs?"
print(f"Instruction: {instruction_1}")
response_1 = generate_response(instruction_1)
print(f"Response: {response_1}")
print()

# Test 2: Instruction with input context
instruction_2 = "Summarize the following text in one sentence:"
input_text_2 = "Fine tunning LLM is a process of adapting a pre-trained language model to specific tasks or domains by updating only a small subset of its parameters, often using techniques like LoRA (Low-Rank Adaptation) to achieve efficient training on consumer hardware."
print(f"Instruction: {instruction_2}")
print(f"Input: {input_text_2}")
response_2 = generate_response(instruction_2, input_text_2)
print(f"Response: {response_2}")
print()

# Test 3: Creative task
instruction_3 = "Write a script for a high school comedy performance involving a song style: rock and roll."
print(f"Instruction: {instruction_3}")
response_3 = generate_response(instruction_3)
print(f"Response: {response_3}")
print()

print("=" * 80)
print("Fine-tuning complete! Model successfully learned to follow instructions.")

Testing fine-tuned Gemma 2B on Alpaca instructions!

Instruction: What is EOS token in LLMs?
Response: EOS token is a token that is used to purchase tokens on the EOS network. It is a digital currency that is used to pay for transactions on the EOS network. EOS tokens are ERC20 tokens, which are tokens that are traded on the Ethereum blockchain. They are used to pay for transactions on the EOS network, and they are also used to pay for services on the EOS network. EOS tokens are also used to incentivize users for providing services on the EOS network. They are an important part of the EOS ecosystem and are used for a variety of purposes. EOS tokens are a great way to earn rewards for providing services on the EOS network. They are also used to pay for transactions and services on the EOS network. EOS tokens are an important part of the EOS ecosystem and are used for a variety of purposes. They are an important part of the EOS ecosystem and are used for a variety of purposes. They are a

In [26]:
# ============================================================================
# STEP 9: SAVE AND LOAD THE FINE-TUNED MODEL
# ============================================================================
# Saving the model allows us to:
# - Share the fine-tuned model with others
# - Use it later without re-training
# - Deploy it to production
# ============================================================================

# Save the LoRA adapter weights
# Note: We only save the adapter (trainable parts), not the entire model
# This is much smaller (MB vs GB) and efficient to distribute
print("Saving LoRA adapter weights...")
model.save_pretrained("./gemma-2b-alpaca-lora-final")
print("Adapter saved!")

# Save the tokenizer (needed to use the model)
tokenizer.save_pretrained("./gemma-2b-alpaca-lora-final")
print("Tokenizer saved!")

print()
print("To use the fine-tuned model later, you can load it like this:")
print()

example_code = """
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

# Load the fine-tuned model
# This loads the base model + LoRA adapter weights
model = AutoPeftModelForCausalLM.from_pretrained(
    "./gemma-2b-alpaca-lora-final",
    device_map="auto"  # Automatically use GPU if available
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("./gemma-2b-alpaca-lora-final")

# Now you can use generate_response() function from Step 8
# Or write your own inference loop
"""

print(example_code)

Saving LoRA adapter weights...
Adapter saved!
Tokenizer saved!

To use the fine-tuned model later, you can load it like this:


from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

# Load the fine-tuned model
# This loads the base model + LoRA adapter weights
model = AutoPeftModelForCausalLM.from_pretrained(
    "./gemma-2b-alpaca-lora-final",
    device_map="auto"  # Automatically use GPU if available
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("./gemma-2b-alpaca-lora-final")

# Now you can use generate_response() function from Step 8
# Or write your own inference loop



# Summary: Gemma 2B Fine-Tuning with Alpaca Dataset

## What We Learned

### 1. **Instruction Following**
- Alpaca dataset teaches models to follow diverse instructions
- Format matters: Clear separation between instruction → response helps learning

### 2. **LoRA (Parameter-Efficient Fine-Tuning)**
- Trains only 0.1-0.5% of parameters vs full fine-tuning
- Works by adding small "adapter" matrices to key layers
- Enables fine-tuning on consumer GPUs (8-16GB VRAM)

### 3. **Key Hyperparameters & Why**
| Parameter | Value | Reason |
|-----------|-------|--------|
| Learning Rate | 3e-4 | Small models benefit from higher LR than base training |
| Batch Size (effective) | 32 | Larger batches = stable gradients, smaller = faster learning |
| Epochs | 3 | Balance between learning patterns and avoiding overfitting |
| Warmup Steps | 100 | Stabilize training from random initialization |
| LoRA Rank (r) | 16 | Smaller models don't need high-rank updates |

### 4. **Training Process Flow**
```
Raw Alpaca Data
    ↓
Format as Instruction→Response
    ↓
Tokenize (convert text→numbers)
    ↓
Apply LoRA adapter to base model
    ↓
Train with gradient descent
    ↓
Save adapter weights only
```

## Key Insights

**Why LoRA works:**
- Neural network weight updates are often low-rank in practice
- Adding trainable low-rank matrices ≈ Full fine-tuning for many tasks
- Computational savings: 100x faster, 100x less memory

**Why Alpaca for learning:**
- Only 52K examples (manageable size)
- Diverse instruction types (summarization, QA, creative, math, etc.)
- Well-documented and community-friendly
- Original paper: https://crfm.stanford.edu/2023/03/13/alpaca.html

**Batch size strategy:**
- `per_device_batch_size=8` × `gradient_accumulation_steps=4` = effective batch size 32
- This mimics training with batch=32 on limited memory
- More stable training without GPU memory issues

## Next Steps to Improve

1. **Longer training:** Increase epochs to 5-10 (monitor validation loss for overfitting)
2. **Better data:** Create domain-specific instruction datasets
3. **Larger model:** Try Gemma 7B or 27B for better performance
4. **Evaluation:** Benchmark on benchmarks like MMLU, HellaSwag
5. **Quantization:** Use int8 or int4 quantization for deployment

## Troubleshooting

| Issue | Solution |
|-------|----------|
| Out of memory | Reduce `per_device_train_batch_size` to 4 |
| Model not learning | Increase `learning_rate` to 5e-4 or decrease to 1e-4 |
| Overfitting | Reduce epochs, increase `weight_decay`, or add more data |
| Slow training | Use `torch.compile()` or increase gradient accumulation |

## Additional Resources

- **LoRA Paper:** https://arxiv.org/abs/2106.09685
- **Gemma Model Card:** https://huggingface.co/google/gemma-2b
- **Alpaca Dataset:** https://huggingface.co/datasets/tatsu-lab/alpaca
- **PEFT Library:** https://huggingface.co/docs/peft
- **Training Tips:** https://huggingface.co/docs/transformers/training